## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


수백만 개의 특성이 있는 훈련 세트를 가지고 있다면, 확률적 경사 하강법이나 미니배치 경사 하강법을 사용할 수 있다

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

검증 오차가 일정하게 상승되고 있다면 과대접합이 되고 있기 시작한 것이다. 그렇다면 검증 에러가 최솟값에 도달하면 바로 훈련을 중지시키는 조기 종료를 해서 문제를 해결할 수 있다.
학습률을 조정을 할 수 있다.
릿지와 라쏘 회귀 처럼 규제를 사용할 수있다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

훈련 에러와 검증 에러가 거의 비슷하고 매우 높다면, 모델이 훈련 세트에 과소적합되었을 가능성이 높을 것이다. 그래서 지금 모델은 높은 편향을 가진 모델이므로 규제 하이퍼파라미터 alpha를 줄여야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

일반적인 선형 회귀는 규제가 없기 때문에 과적합(overfitting) 되기 쉽습니다. 반면에 릿지 휘기는 규제가 있는 선형 회귀 버전으로 과접합을 방지하고 모델의 일반화 성능을 향상시켜준다.(높은 분산 문제를 해결하고 릿지 회귀는 가중치를 최대한 0에 가깝게 하여 과대접합을 억제할 수 있다.)

릿지 회귀 대신 라쏘 회귀는 가중치 벡터의 l1 노름을 사용하며, 덜 중요한 특성의 가중치를 제거하려한다. 즉, 데이터에 불필요하거나 잡음이 많은 특성이 섞여 있을 경우에 사용하면 좋다.

라쏘 회귀 대신 엘라스틱 넷을 사용해야하는 이유는 다음과 같다. 라쏘는 덜 중요한 특성의 가중치를 제거하려는 반면에 엘라스틱넷(Elastic Net)은 L1과 L2 정규화를 혼합해서 사용하므로, 릿지와 라쏘의 장점을 모두 결합한 형태여서 라쏘의 과도한 규제를 막을 수 있다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [7]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# 1. 데이터 불러오기 및 전처리
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# 정규화
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 원-핫 인코딩
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y)

# 훈련/검증 세트 분리
X_train, X_val, y_train, y_val = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

# 2. 소프트맥스 함수 정의
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# 3. 손실 함수
def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-9), axis=1))

# 4. 예측 함수
def predict(X, W, b):
    logits = np.dot(X, W) + b
    return softmax(logits)

# 5. 하이퍼파라미터 초기화
n_features = X.shape[1]
n_classes = y_onehot.shape[1]

W = np.random.randn(n_features, n_classes) * 0.01
b = np.zeros((1, n_classes))

learning_rate = 0.1
epochs = 500
tolerance = 1e-4
patience = 10
best_val_loss = np.inf
patience_counter = 0

# 6. 학습 루프
for epoch in range(epochs):
    # 예측
    y_pred = predict(X_train, W, b)

    # 손실 계산
    loss = cross_entropy(y_train, y_pred)

    # 기울기 계산
    grad_W = np.dot(X_train.T, (y_pred - y_train)) / len(X_train)
    grad_b = np.mean(y_pred - y_train, axis=0, keepdims=True)

    # 가중치 업데이트
    W -= learning_rate * grad_W
    b -= learning_rate * grad_b

    # 검증 손실
    val_pred = predict(X_val, W, b)
    val_loss = cross_entropy(y_val, val_pred)

    print(f"Epoch {epoch+1}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}")

    # 조기 종료 조건 확인
    if val_loss < best_val_loss - tolerance:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("조기 종료")
            break

# 7. 정확도 평가
def accuracy(y_true, y_pred_probs):
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    return np.mean(y_true_labels == y_pred_labels)

train_acc = accuracy(y_train, predict(X_train, W, b))
val_acc = accuracy(y_val, predict(X_val, W, b))

print(f"최종 Train Accuracy: {train_acc:.4f}")
print(f"최종 Validation Accuracy: {val_acc:.4f}")

Epoch 1, Train Loss: 1.1016, Val Loss: 1.0049
Epoch 2, Train Loss: 1.0113, Val Loss: 0.9269
Epoch 3, Train Loss: 0.9369, Val Loss: 0.8623
Epoch 4, Train Loss: 0.8755, Val Loss: 0.8087
Epoch 5, Train Loss: 0.8246, Val Loss: 0.7637
Epoch 6, Train Loss: 0.7820, Val Loss: 0.7256
Epoch 7, Train Loss: 0.7461, Val Loss: 0.6931
Epoch 8, Train Loss: 0.7155, Val Loss: 0.6649
Epoch 9, Train Loss: 0.6892, Val Loss: 0.6404
Epoch 10, Train Loss: 0.6663, Val Loss: 0.6188
Epoch 11, Train Loss: 0.6462, Val Loss: 0.5997
Epoch 12, Train Loss: 0.6284, Val Loss: 0.5826
Epoch 13, Train Loss: 0.6126, Val Loss: 0.5672
Epoch 14, Train Loss: 0.5984, Val Loss: 0.5533
Epoch 15, Train Loss: 0.5855, Val Loss: 0.5406
Epoch 16, Train Loss: 0.5738, Val Loss: 0.5290
Epoch 17, Train Loss: 0.5631, Val Loss: 0.5183
Epoch 18, Train Loss: 0.5533, Val Loss: 0.5084
Epoch 19, Train Loss: 0.5442, Val Loss: 0.4992
Epoch 20, Train Loss: 0.5358, Val Loss: 0.4906
Epoch 21, Train Loss: 0.5280, Val Loss: 0.4826
Epoch 22, Train Loss: 